In [1]:
import gzip
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'experiments.ipynb').exists() and (NOTEBOOK_DIR / 'lab01' / 'experiments.ipynb').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'lab01'
CORPUS_PATH = NOTEBOOK_DIR.parent / 'c4-train.00000-of-01024-30K.json.gz'
RESULTS_PATH = NOTEBOOK_DIR / 'results.csv'
if not CORPUS_PATH.exists():
    raise FileNotFoundError(f'Corpus not found: {CORPUS_PATH.resolve()}')

documents = []
with gzip.open(CORPUS_PATH, 'rt', encoding='utf-8') as corpus_file:
    for line in corpus_file:
        record = json.loads(line)
        text = record.get('text', '').strip()
        if text:
            documents.append(text)
        if len(documents) == 30000:
            break

doc_ids = [f'D{index:05d}' for index in range(len(documents))]
print('N documents:', len(documents))
print('Example ID:', doc_ids[0])
print('Example preview:', documents[0][:200])

N documents: 30000
Example ID: D00000
Example preview: Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class


## Part D: Inspect the Sparse Representation



In [2]:
vectorizer = CountVectorizer(lowercase=True, min_df=1, token_pattern=r'(?u)\b\w\w+\b')
counts = vectorizer.fit_transform(documents)
terms = np.array(vectorizer.get_feature_names_out())
tf = normalize(counts, norm='l1', axis=1)
tfidf_transformer = TfidfTransformer(norm='l2', use_idf=True, smooth_idf=True, sublinear_tf=False)
X = tfidf_transformer.fit_transform(tf)
N, V = X.shape
nnz = X.nnz
sparsity = 1 - nnz / (N * V)
print('N =', N)
print('V =', V)
print('Matrix shape =', X.shape)
print('nnz =', nnz)
print('Sparsity =', sparsity)
print('Sparsity (%) =', 100 * sparsity)

df = np.asarray(counts.getnnz(axis=0)).ravel()
top_df_idx = np.argsort(-df, kind='stable')[:20]
top_idf_idx = np.argsort(-tfidf_transformer.idf_, kind='stable')[:20]
selected_doc_idx = 0
row = X.getrow(selected_doc_idx).toarray().ravel()
nonzero_tfidf_idx = np.flatnonzero(row > 0)
top_tfidf_idx = nonzero_tfidf_idx[np.argsort(-row[nonzero_tfidf_idx], kind='stable')[:20]]

top_df_table = pd.DataFrame({'term': terms[top_df_idx], 'document_frequency': df[top_df_idx]})
top_idf_table = pd.DataFrame({'term': terms[top_idf_idx], 'idf': tfidf_transformer.idf_[top_idf_idx]})
top_tfidf_table = pd.DataFrame({'term': terms[top_tfidf_idx], 'tfidf': row[top_tfidf_idx]})
print('Top 20 by DF'); display(top_df_table)
print('Top 20 by IDF'); display(top_idf_table)
print(f'Top 20 TF-IDF in {doc_ids[selected_doc_idx]}'); display(top_tfidf_table)

N = 30000
V = 193540
Matrix shape = (30000, 193540)
nnz = 4985822
Sparsity = 0.9991412934449382
Sparsity (%) = 99.91412934449382
Top 20 by DF


,term,document_frequency
0,the,27893
1,and,27423
2,to,26689
3,of,26031
4,in,25224
5,for,23651
6,is,22739
7,with,21405
8,on,20262
9,that,18370


Top 20 by IDF


,term,idf
0,00000,10.615839
1,000000,10.615839
2,00000000,10.615839
3,0000000000000965,10.615839
4,00000001,10.615839
5,00000048,10.615839
6,000002,10.615839
7,000004b926f1,10.615839
8,00000781,10.615839
9,0000085054,10.615839


Top 20 TF-IDF in D00000


,term,tfidf
0,bbq,0.461808
1,class,0.275842
2,meat,0.194158
3,balay,0.179403
4,kcbs,0.172551
5,lonestar,0.167689
6,will,0.154676
7,missoula,0.150594
8,apron,0.141357
9,smoker,0.140491


## Part F: Preprocessing Ablation


In [3]:
def normalize_punctuation(text):
    return re.sub(r'[^\w\s]', ' ', text.lower())

pipelines = {
    'A Minimal': CountVectorizer(lowercase=True, min_df=1, token_pattern=r'(?u)\b\w\w+\b'),
    'B Normalized': CountVectorizer(preprocessor=normalize_punctuation, stop_words='english', min_df=1, token_pattern=r'(?u)\b\w\w+\b'),
    'C Extended': CountVectorizer(preprocessor=normalize_punctuation, analyzer='char_wb', ngram_range=(3, 5), max_features=50000, min_df=1),
}

pipeline_objects = {}
pipeline_rows = []
for name, vec in pipelines.items():
    pipeline_counts = vec.fit_transform(documents)
    pipeline_tf = normalize(pipeline_counts, norm='l1', axis=1)
    transformer = TfidfTransformer(norm='l2', use_idf=True, smooth_idf=True)
    matrix = transformer.fit_transform(pipeline_tf)
    analyzer = vec.build_analyzer()
    avg_units = np.mean([len(analyzer(document)) for document in documents])
    matrix_sparsity = 1 - matrix.nnz / (matrix.shape[0] * matrix.shape[1])
    pipeline_objects[name] = {'vectorizer': vec, 'transformer': transformer, 'counts': pipeline_counts, 'tf': pipeline_tf, 'matrix': matrix}
    pipeline_rows.append({'pipeline': name, 'vocabulary_size': matrix.shape[1], 'average_emitted_units_per_document': avg_units, 'matrix_sparsity': matrix_sparsity})
pipeline_summary = pd.DataFrame(pipeline_rows)
display(pipeline_summary)

,pipeline,vocabulary_size,average_emitted_units_per_document,matrix_sparsity
0,A Minimal,193540,351.338800,0.999141
1,B Normalized,193226,190.132600,0.999367
2,C Extended,50000,4095.691967,0.967260


In [4]:
example_queries = ['medical image classification', 'transformer language model', 'deep learning healthcare', 'natural language processing']
def query_units(query, vectorizer):
    return vectorizer.build_analyzer()(query)

oov_rows = []
for name, objects in pipeline_objects.items():
    vocabulary = set(objects['vectorizer'].get_feature_names_out())
    units = [unit for query in example_queries for unit in query_units(query, objects['vectorizer'])]
    oov_rate = sum(unit not in vocabulary for unit in units) / len(units) if units else 0.0
    oov_rows.append({'pipeline': name, 'oov_rate_on_example_queries': oov_rate})
f_summary = pipeline_summary.merge(pd.DataFrame(oov_rows), on='pipeline')
display(f_summary)
print('Add mean P@5 after entering the same manual qrels used in Part H.')

,pipeline,vocabulary_size,average_emitted_units_per_document,matrix_sparsity,oov_rate_on_example_queries
0,A Minimal,193540,351.338800,0.999141,0.0
1,B Normalized,193226,190.132600,0.999367,0.0
2,C Extended,50000,4095.691967,0.967260,0.0


Add mean P@5 after entering the same manual qrels used in Part H.


In [5]:
# Diagnostic controls to isolate lowercasing, punctuation normalization, and stopword handling.
case_preserved = CountVectorizer(lowercase=False, min_df=1, token_pattern=r'(?u)\b\w\w+\b')
punctuation_only = CountVectorizer(preprocessor=normalize_punctuation, min_df=1, token_pattern=r'(?u)\b\w\w+\b')
diagnostic_vectors = {
    'Lowercase, no stopwords': pipeline_objects['A Minimal']['vectorizer'],
    'Case-preserving, no stopwords': case_preserved,
    'Punctuation normalized, no stopwords': punctuation_only,
    'Punctuation normalized, English stopwords': pipeline_objects['B Normalized']['vectorizer'],
}
diagnostic_rows = []
for label, diagnostic_vectorizer in diagnostic_vectors.items():
    if not hasattr(diagnostic_vectorizer, 'vocabulary_'):
        diagnostic_vectorizer.fit(documents)
    diagnostic_analyzer = diagnostic_vectorizer.build_analyzer()
    diagnostic_rows.append({
        'controlled_setting': label,
        'vocabulary_size': len(diagnostic_vectorizer.vocabulary_),
        'average_emitted_units_per_document': np.mean([len(diagnostic_analyzer(document)) for document in documents]),
    })
display(pd.DataFrame(diagnostic_rows))

,controlled_setting,vocabulary_size,average_emitted_units_per_document
0,"Lowercase, no stopwords",193540,351.338800
1,"Case-preserving, no stopwords",240444,351.338467
2,"Punctuation normalized, no stopwords",193540,351.338800
3,"Punctuation normalized, English stopwords",193226,190.132600


## Part G: Document Search Engine

In [6]:
def rank_documents(query, vectorizer, transformer, document_matrix):
    query_counts = vectorizer.transform([query])
    query_tf = normalize(query_counts, norm='l1', axis=1)
    query_vector = transformer.transform(query_tf)
    scores = cosine_similarity(query_vector, document_matrix).ravel()
    return scores, np.argsort(-scores, kind='stable')

def search(query, vectorizer, transformer, document_matrix, document_ids, document_texts, top_k=5):
    scores, order = rank_documents(query, vectorizer, transformer, document_matrix)
    order = order[:top_k]
    return pd.DataFrame({
        'Rank': np.arange(1, len(order) + 1),
        'Document ID': [document_ids[index] for index in order],
        'Similarity': scores[order],
        'Document preview': [document_texts[index][:240].replace('\n', ' ') for index in order],
    })

chosen = pipeline_objects['A Minimal']
for query in example_queries:
    print('Query:', query)
    display(search(query, chosen['vectorizer'], chosen['transformer'], chosen['matrix'], doc_ids, documents, top_k=5))

Query: medical image classification


,Rank,Document ID,Similarity,Document preview
0,1,D18971,0.400030,The new RTS Environmental Classification syste...
1,2,D08527,0.349998,History of maize classification. How races use...
2,3,D19908,0.253427,Download League Of Legends Wallpapers in high-...
3,4,D17794,0.240507,Filters the output of 'wp_calculate_image_size...
4,5,D12658,0.233145,"This guidance is for pharmacists who handle, u..."


Query: transformer language model


,Rank,Document ID,Similarity,Document preview
0,1,D27936,0.471849,"hi, I am having problems with transformer / ci..."
1,2,D25428,0.279103,"Note: If you're on an iPhone, you cannot chang..."
2,3,D04075,0.222687,Looking for Spanish language instructor to imp...
3,4,D00701,0.216096,Program in Teaching French as a Foreign Langua...
4,5,D13690,0.213617,Text in finnish language missing. Sorry for th...


Query: deep learning healthcare


,Rank,Document ID,Similarity,Document preview
0,1,D09252,0.278174,"SAN DIEGO AND WASHINGTON, D.C. – Sept. 5, 2018..."
1,2,D06123,0.275276,"With today’s advancement in technology, it is ..."
2,3,D11119,0.274719,The opportunities offered by Big Data will onl...
3,4,D11979,0.273027,Doctorate of Healthcare Organization Program i...
4,5,D07564,0.240487,"would I have learned, learnt? would you have l..."


Query: natural language processing


,Rank,Document ID,Similarity,Document preview
0,1,D25428,0.362266,"Note: If you're on an iPhone, you cannot chang..."
1,2,D08705,0.341789,These regulations may be called the Food Safet...
2,3,D04075,0.289040,Looking for Spanish language instructor to imp...
3,4,D05699,0.282915,"Truth-value gaps in natural language. Waldo, J..."
4,5,D00701,0.280485,Program in Teaching French as a Foreign Langua...


## Part H: Evaluation

Manually inspect retrieved documents and replace the empty dictionary below with 5-10 queries and reviewed relevant IDs. Do not generate relevance from term overlap. The notebook deliberately refuses to report official metrics while qrels are empty.

In [7]:
qrels = {
    "Q1": {"query": "MRI diagnostic imaging", "relevant": {"D17055"}},
    "Q2": {"query": "text categorization", "relevant": {"D11626"}},
    "Q3": {"query": "clinical outcome prediction", "relevant": {"D04653"}},
    "Q4": {"query": "natural language processing", "relevant": {"D05913"}},
    "Q5": {"query": "barbecue cooking class", "relevant": {"D00001"}},
    "Q6": {"query": "wild garlic pesto shrimp pasta", "relevant": {"D01288"}},
    "Q7": {"query": "Chateau Latour 1996 wine", "relevant": {"D09252"}},
    "Q8": {"query": "carpenter ant identification", "relevant": {"D26462"}},
    "Q9": {"query": "image overlay manipulation", "relevant": {"D27936"}},
    "Q10": {"query": "wildflower wall hanging kit", "relevant": {"D18971"}},
}

def evaluate_one_query(query, relevant_ids, vectorizer, transformer, document_matrix, document_ids, document_texts, k=5):
    result = search(query, vectorizer, transformer, document_matrix, document_ids, document_texts, top_k=k)
    retrieved = result['Document ID'].tolist()
    relevant_retrieved = sum(document_id in relevant_ids for document_id in retrieved)
    precision_at_k = relevant_retrieved / k
    recall_at_k = relevant_retrieved / len(relevant_ids) if relevant_ids else 0.0
    _, full_order = rank_documents(query, vectorizer, transformer, document_matrix)
    first_relevant_rank = next((rank for rank, index in enumerate(full_order, start=1) if document_ids[index] in relevant_ids), None)
    reciprocal_rank = 1 / first_relevant_rank if first_relevant_rank is not None else 0.0
    return result, {'P@5': precision_at_k, 'Recall@5': recall_at_k, 'RR': reciprocal_rank}

if not qrels:
    print('qrels is empty: add 5-10 manually judged queries before reporting Part H metrics.')
else:
    if not 5 <= len(qrels) <= 10:
        raise ValueError('W1 requires 5-10 evaluation queries.')
    if any(not item.get('relevant') for item in qrels.values()):
        raise ValueError('Each query needs at least one manually judged relevant document.')
    known_ids = set(doc_ids)
    for query_id, item in qrels.items():
        unknown_ids = set(item['relevant']) - known_ids
        if unknown_ids:
            raise ValueError(f'{query_id} contains unknown document IDs: {sorted(unknown_ids)}')
    rows = []
    metrics = []
    for pipeline_name, objects in pipeline_objects.items():
        for query_id, item in qrels.items():
            result, metric = evaluate_one_query(item['query'], item['relevant'], objects['vectorizer'], objects['transformer'], objects['matrix'], doc_ids, documents)
            metrics.append({'pipeline': pipeline_name, 'query_id': query_id, **metric})
            for _, row in result.iterrows():
                rows.append({'pipeline': pipeline_name, 'query_id': query_id, 'query': item['query'], 'rank': int(row['Rank']), 'document_id': row['Document ID'], 'similarity': row['Similarity'], 'relevant': row['Document ID'] in item['relevant'], **metric})
    results = pd.DataFrame(rows)
    results.to_csv(RESULTS_PATH, index=False)
    metrics_table = pd.DataFrame(metrics)
    mean_metrics = metrics_table.groupby('pipeline')[['P@5', 'Recall@5', 'RR']].mean().rename(columns={'RR': 'MRR'})
    f_summary = f_summary.merge(mean_metrics[['P@5']].rename(columns={'P@5': 'mean_P@5'}), left_on='pipeline', right_index=True, how='left')
    print('P@5 = relevant retrieved in top 5 / 5')
    print('Recall@5 = relevant retrieved in top 5 / manually judged relevant documents')
    print('MRR = mean reciprocal rank of the first relevant result in the full ranking; no relevant result contributes 0')
    print('Per-pipeline macro metrics:')
    display(mean_metrics)
    print('Part F comparison, including search performance:')
    display(f_summary)
    display(results)

P@5 = relevant retrieved in top 5 / 5
Recall@5 = relevant retrieved in top 5 / manually judged relevant documents
MRR = mean reciprocal rank of the first relevant result in the full ranking; no relevant result contributes 0
Per-pipeline macro metrics:


,P@5,Recall@5,MRR
pipeline,,,
A Minimal,0.0,0.0,0.000179
B Normalized,0.0,0.0,0.000179
C Extended,0.0,0.0,0.000065


Part F comparison, including search performance:


,pipeline,vocabulary_size,average_emitted_units_per_document,matrix_sparsity,oov_rate_on_example_queries,mean_P@5
0,A Minimal,193540,351.338800,0.999141,0.0,0.0
1,B Normalized,193226,190.132600,0.999367,0.0,0.0
2,C Extended,50000,4095.691967,0.967260,0.0,0.0


,pipeline,query_id,query,rank,document_id,similarity,relevant,P@5,Recall@5,RR
0,A Minimal,Q1,MRI diagnostic imaging,1,D17054,0.462844,False,0.0,0.0,0.000058
1,A Minimal,Q1,MRI diagnostic imaging,2,D11697,0.241457,False,0.0,0.0,0.000058
2,A Minimal,Q1,MRI diagnostic imaging,3,D19561,0.203813,False,0.0,0.0,0.000058
3,A Minimal,Q1,MRI diagnostic imaging,4,D29972,0.198964,False,0.0,0.0,0.000058
4,A Minimal,Q1,MRI diagnostic imaging,5,D18221,0.195080,False,0.0,0.0,0.000058
...,...,...,...,...,...,...,...,...,...,...
145,C Extended,Q10,wildflower wall hanging kit,1,D10246,0.233007,False,0.0,0.0,0.000045
146,C Extended,Q10,wildflower wall hanging kit,2,D18970,0.218506,False,0.0,0.0,0.000045
147,C Extended,Q10,wildflower wall hanging kit,3,D01732,0.211901,False,0.0,0.0,0.000045
148,C Extended,Q10,wildflower wall hanging kit,4,D08414,0.204655,False,0.0,0.0,0.000045
